In [24]:
!pip install google-genai langchain faiss-cpu python-dotenv
!pip install langchain[all]
!pip install sentence-transformers
!pip install datasets huggingface-hub



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [ ]:
import requests as rq
import bs4 
import json
from dotenv import load_dotenv
import langchain as lc
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI


SyntaxError: invalid syntax (4040810655.py, line 8)

In [9]:
import requests as rq
from bs4 import BeautifulSoup
import json

def _search_google(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://www.google.com/search",
        params={"q": search, "hl": "en"},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select("div.g")[:5]:
        title_tag = item.select_one("h3")
        link_tag = item.select_one("a")
        snippet_tag = item.select_one(".VwiC3b, .IsZvec")
        if title_tag and link_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": link_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_duckduckgo(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://html.duckduckgo.com/html",
        params={"q": search},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select(".result"):
        title_tag = item.select_one("a.result__a")
        snippet_tag = item.select_one("a.result__snippet, .result__snippet")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def _search_bing(search):
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/124.0.0.0 Safari/537.36"
        )
    }
    resp = rq.get(
        "https://www.bing.com/search",
        params={"q": search},
        headers=headers,
        timeout=10,
    )
    soup = BeautifulSoup(resp.text, "html.parser")
    results = []
    for item in soup.select("li.b_algo"):
        title_tag = item.select_one("h2 a")
        snippet_tag = item.select_one("p")
        if title_tag:
            results.append({
                "title": title_tag.get_text(strip=True),
                "link": title_tag.get("href"),
                "snippet": snippet_tag.get_text(strip=True) if snippet_tag else "",
            })
    return results

def search_engine(search):
    for search_fn in (_search_google, _search_duckduckgo, _search_bing):
        results = search_fn(search)
        if results:
            return json.dumps({"search": search, "results": results}, ensure_ascii=False)

    return json.dumps({
        "search": search,
        "results": [],
        "error": "No results returned by Google, DuckDuckGo, or Bing.",
    }, ensure_ascii=False)

print(search_engine("nice"))


{"search": "nice", "results": [{"title": "NiCE: Global Locations", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fwww.nice.com%2Fcompany%2Fglobal%2Dlocations&rut=e6ed01fd918abcd26d09002eaf9fbc1dea5ebcf5ba5ff4c48c1410922fdcb679", "snippet": "ExploreNICE'sglobal presence and discover their worldwide locations offering innovative solutions to enhance customer experience and business operations."}, {"title": "NICE Definition & Meaning - Merriam-Webster", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fwww.merriam%2Dwebster.com%2Fdictionary%2Fnice&rut=4657f9c70ad33593402d8333bf7f1f6a932d143b1fd3064fb379c8a2b9ffdbb0", "snippet": "The meaning ofNICEis polite, kind. How to usenicein a sentence. Synonym Discussion ofNice."}, {"title": "Nice - Wikipedia", "link": "//duckduckgo.com/l/?uddg=https%3A%2F%2Fen.wikipedia.org%2Fwiki%2FNice&rut=5e8796f53bdb0b3e51eb90728c884caa169f0a7c721f9b0480075c68549dd06a", "snippet": "Nice[a] (/ niːs / NEESS; French pronunciation: [nis] ⓘ) is a city located in the 

In [ ]:
hardcoded =[
    "https://bookchapter.org/kitaplar/Medical%20Diagnosis%20and%20Treatment%20Methods%20in%20Basic%20Medical%20Sciences.pdf",
    "https://applications.emro.who.int/dsaf/dsa236.pdf" 
]

In [ ]:



def load_pdf(url):
    loader = PyPDFLoader(url)
    return loader.load()



In [ ]:
def split_text(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    return text_splitter.split_documents(documents)

In [ ]:
userquery = "What are the symptoms of diabetes?"


In [ ]:
# Hugging Face collections are not directly loadable with datasets.load_dataset.
# Use a specific dataset repo ID or download files from huggingface_hub instead.


In [26]:
from datasets import load_dataset

dataset = load_dataset(
    "openlifescienceai/medmcqa",
    split="train"
)

print(dataset)
print(dataset[0])

Generating validation split: 100%|██████████| 4183/4183 [00:00<00:00, 167490.27 examples/s]


Dataset({
    features: ['id', 'question', 'opa', 'opb', 'opc', 'opd', 'cop', 'choice_type', 'exp', 'subject_name', 'topic_name'],
    num_rows: 182822
})
{'id': 'e9ad821a-c438-4965-9f77-760819dfa155', 'question': 'Chronic urethral obstruction due to benign prismatic hyperplasia can lead to the following change in kidney parenchyma', 'opa': 'Hyperplasia', 'opb': 'Hyperophy', 'opc': 'Atrophy', 'opd': 'Dyplasia', 'cop': 2, 'choice_type': 'single', 'exp': 'Chronic urethral obstruction because of urinary calculi, prostatic hyperophy, tumors, normal pregnancy, tumors, uterine prolapse or functional disorders cause hydronephrosis which by definition is used to describe dilatation of renal pelvis and calculus associated with progressive atrophy of the kidney due to obstruction to the outflow of urine Refer Robbins 7yh/9,1012,9/e. P950', 'subject_name': 'Anatomy', 'topic_name': 'Urinary tract'}


In [ ]:
for i in dataset:
    print(i)